# Udfordring: Analyse af tekst om datalogi

I dette eksempel laver vi en simpel øvelse, der dækker alle trin i en traditionel data science-proces. Du behøver ikke at skrive noget kode, du kan bare klikke på cellerne nedenfor for at udføre dem og observere resultatet. Som en udfordring opfordres du til at prøve denne kode med forskellige data.

## Mål

I denne lektion har vi diskuteret forskellige begreber relateret til Data Science. Lad os prøve at opdage flere relaterede begreber ved at lave noget **tekstudvinding**. Vi starter med en tekst om Data Science, udtrækker nøgleord fra den, og prøver derefter at visualisere resultatet.

Som tekst vil jeg bruge siden om Data Science fra Wikipedia:


In [ ]:
url = 'https://en.wikipedia.org/wiki/Data_science'

## Trin 1: Indhent dataene

Det første trin i enhver datavidenskabsproces er at hente dataene. Vi vil bruge `requests` biblioteket til det:


In [ ]:
import requests

# Define a custom header.
headers = {
    'User-Agent': 'DataScienceChallenge/1.0 (myemail@gmail.com)'
}

# Pass the headers into the get request
response = requests.get(url, headers=headers)

if response.status_code == 200:
    text = response.content.decode('utf-8')
    print(text[:1000])
else:
    print(f"Error: {response.status_code}")

## Trin 2: Transformering af dataene

Det næste trin er at konvertere dataene til den form, der er egnet til behandling. I vores tilfælde har vi downloadet HTML-kildekode fra siden, og vi skal konvertere den til almindelig tekst.

Der findes mange måder at gøre dette på. Vi vil bruge [BeautifulSoup](https://www.crummy.com/software/BeautifulSoup/), et populært Python-bibliotek til parsing af HTML. BeautifulSoup gør det muligt for os at målrette specifikke HTML-elementer, så vi kan fokusere på hovedartiklens indhold fra Wikipedia og reducere nogle navigationsmenuer, sidebjælker, sidefødder og andet irrelevant indhold (selvom noget standardtekst stadig kan være tilbage).


Først skal vi installere BeautifulSoup-biblioteket til HTML-parsing:


In [ ]:
import sys
!{sys.executable} -m pip install beautifulsoup4

In [ ]:
from bs4 import BeautifulSoup

# Parse the HTML content
soup = BeautifulSoup(text, 'html.parser')

# Extract only the main article content from Wikipedia
# Wikipedia uses 'mw-parser-output' class for the main article content
content = soup.find('div', class_='mw-parser-output')

def clean_wikipedia_content(content_node):
    """Remove common non-article elements from a Wikipedia content node."""
    # Strip jump links, navboxes, reference lists/superscripts, edit sections, TOC, sidebars, etc.
    selectors = [
        '.mw-jump-link',
        '.navbox',
        '.reflist',
        'sup.reference',
        '.mw-editsection',
        '.hatnote',
        '.metadata',
        '.infobox',
        '#toc',
        '.toc',
        '.sidebar',
    ]
    for selector in selectors:
        for el in content_node.select(selector):
            el.decompose()

if content:
    # Clean the content node to better approximate article text only.
    clean_wikipedia_content(content)
    text = content.get_text(separator=' ', strip=True)
    print(text[:1000])
else:
    print("Could not find main content. Using full page text.")
    text = soup.get_text(separator=' ', strip=True)
    print(text[:1000])

## Trin 3: Få indsigt

Det vigtigste trin er at omdanne vores data til en form, hvorfra vi kan udlede indsigt. I vores tilfælde vil vi udtrække nøgleord fra teksten og se, hvilke nøgleord der er mere meningsfulde.

Vi bruger Python-biblioteket kaldet [RAKE](https://github.com/aneesha/RAKE) til nøgleordsudtrækning. Først lad os installere dette bibliotek, hvis det ikke allerede er til stede: 


In [ ]:
import sys
!{sys.executable} -m pip install nlp_rake

Hovedfunktionen er tilgængelig fra `Rake`-objektet, som vi kan tilpasse ved hjælp af nogle parametre. I vores tilfælde sætter vi minimumslængden for et søgeord til 5 tegn, minimumsfrekvensen for et søgeord i dokumentet til 3, og maksimalt antal ord i et søgeord - til 2. Du er velkommen til at eksperimentere med andre værdier og observere resultatet.


In [ ]:
import nlp_rake
extractor = nlp_rake.Rake(max_words=2,min_freq=3,min_chars=5)
res = extractor.apply(text)
res


Vi har fået en liste over termer sammen med tilhørende grad af vigtighed. Som du kan se, er de mest relevante discipliner, såsom maskinlæring og big data, til stede i listen på topplaceringerne.

## Trin 4: Visualisering af resultatet

Folk kan bedst fortolke data i visuel form. Derfor giver det ofte mening at visualisere dataene for at opnå nogle indsigter. Vi kan bruge `matplotlib`-biblioteket i Python til at plotte simpel fordeling af søgeordene med deres relevans:


In [ ]:
import matplotlib.pyplot as plt

def plot(pair_list):
    k,v = zip(*pair_list)
    plt.bar(range(len(k)),v)
    plt.xticks(range(len(k)),k,rotation='vertical')
    plt.show()

plot(res)

Der er dog en endnu bedre måde at visualisere ordhyppigheder på – ved at bruge **Word Cloud**. Vi bliver nødt til at installere et andet bibliotek for at plotte word cloud fra vores nøgleordsliste.


In [ ]:
!{sys.executable} -m pip install wordcloud

`WordCloud`-objektet er ansvarligt for at tage enten original tekst eller en forudberegnet liste over ord med deres frekvenser og returnere et billede, som derefter kan vises ved hjælp af `matplotlib`:


In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

wc = WordCloud(background_color='white',width=800,height=600)
plt.figure(figsize=(15,7))
plt.imshow(wc.generate_from_frequencies({ k:v for k,v in res }))

Vi kan også give den originale tekst til `WordCloud` - lad os se, om vi kan få et lignende resultat:


In [ ]:
plt.figure(figsize=(15,7))
plt.imshow(wc.generate(text))

In [ ]:
wc.generate(text).to_file('images/ds_wordcloud.png')

Du kan se, at ordskyen nu ser mere imponerende ud, men den indeholder også meget støj (f.eks. uvedkommende ord som `Retrieved on`). Derudover får vi færre nøgleord, der består af to ord, såsom *data scientist* eller *computer science*. Dette skyldes, at RAKE-algoritmen klarer et meget bedre arbejde med at udvælge gode nøgleord fra teksten. Dette eksempel illustrerer vigtigheden af datapræbehandling og oprydning, fordi et klart billede til sidst vil give os mulighed for at træffe bedre beslutninger.

I denne øvelse har vi gennemgået en simpel proces til at udtrække noget mening fra Wikipedia-tekst i form af nøgleord og ordsky. Dette eksempel er ret enkelt, men det demonstrerer godt alle de typiske trin, en dataforsker vil tage, når han arbejder med data, fra dataindsamling til visualisering.

I vores kursus vil vi diskutere alle disse trin i detaljer. 


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfraskrivelse**:
Dette dokument er blevet oversat ved hjælp af AI-oversættelsestjenesten [Co-op Translator](https://github.com/Azure/co-op-translator). Selvom vi bestræber os på nøjagtighed, skal du være opmærksom på, at automatiserede oversættelser kan indeholde fejl eller unøjagtigheder. Det originale dokument på dets oprindelige sprog bør betragtes som den autoritative kilde. For kritisk information anbefales professionel menneskelig oversættelse. Vi påtager os intet ansvar for misforståelser eller fejltolkninger, der opstår som følge af brugen af denne oversættelse.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
